# 03 · Modelos

Ordinal, riesgo discreto, gradient boosting y Transformer; calibración en el 20 % por vehículo.

In [ ]:
%load_ext autoreload
%autoreload 2
import sys; sys.path.append('..')
import gc
import pandas as pd
import utils

ID, T = utils.ID, utils.T

In [ ]:
points = {
    'train': pd.concat([utils.load_processed('train_labels_conv'),
                        utils.load_processed('train_labels_landmark')])[[ID, T]],
    'validation': utils.load_processed('validation_eval')[[ID, T]],
    'test': utils.load_processed('test_eval')[[ID, T]],
}

for split, pts in points.items():
    ro = utils.load_processed(f'{split}_readouts')
    F = utils.build_features(ro, pts)
    del ro; gc.collect()

    spec = utils.load_processed(f'{split}_specifications')
    spec = spec.astype({c: 'category' for c in spec.columns if c.startswith('Spec_')})
    F = F.merge(spec, on=ID, how='left')

    utils.to_parquet(F, f'{split}_features')
    print(split, F.shape, '| puntos pedidos:', len(pts.drop_duplicates()))
    del F; gc.collect()

In [ ]:
import numpy as np

F    = utils.load_processed('train_features')
conv = utils.load_processed('train_labels_conv')
lm   = utils.load_processed('train_labels_landmark')

data = {
    'convencional': F.merge(conv[[ID, T, 'y_conv']], on=[ID, T]).rename(columns={'y_conv': 'y'}),
    'landmarking':  F.merge(lm.loc[lm['y_cens'].notna(), [ID, T, 'y_cens']], on=[ID, T])
                     .rename(columns={'y_cens': 'y'}),
}
for d in data.values():
    d['y'] = d['y'].astype(int)

cal_ids = utils.calibration_split(F[ID])          # 20 % de vehículos para calibración
feats = [c for c in F.columns if c != ID]          # time_step se queda: es la edad

Fv = utils.load_processed('validation_features').merge(
        utils.load_processed('validation_eval')[[ID, T, 'class_label']], on=[ID, T])
Xv, yv = Fv[feats], Fv['class_label'].to_numpy()

for name, d in data.items():
    fit = d[~d[ID].isin(cal_ids)]
    print(f'{name}: {len(fit)} filas de entrenamiento, {len(d) - len(fit)} de calibración')

In [ ]:
models, res = {}, []

for name, d in data.items():
    fit = d[~d[ID].isin(cal_ids)]
    b = utils.train_lgbm(fit[feats], fit['y'], Xv, yv)
    models[name] = b
    P = b.predict(Xv, num_iteration=b.best_iteration)
    for rule, decide in [('argmax', utils.decide_argmax), ('min_costo', utils.decide_min_cost)]:
        yhat = decide(P)
        res.append({'formulación': name, 'regla': rule,
                    'costo_total': utils.total_cost(yv, yhat),
                    '% a revisión': round(100 * (yhat > 0).mean(), 1),
                    'iteraciones': b.best_iteration})

for c in [0, 4]:
    yhat = np.full(len(yv), c)
    res.append({'formulación': 'trivial', 'regla': f'siempre {c}',
                'costo_total': utils.total_cost(yv, yhat),
                '% a revisión': 100.0 * (c > 0)})

pd.DataFrame(res)

In [ ]:
from sklearn.metrics import roc_auc_score, accuracy_score
import lightgbm as lgb

# --- 1. ¿Mejora el COSTO con más árboles aunque el log-loss empeore?
d = data['convencional']
fit = d[~d[ID].isin(cal_ids)]
p = dict(objective='multiclass', num_class=5, learning_rate=0.05, num_leaves=31,
         min_data_in_leaf=50, feature_fraction=0.5, bagging_fraction=0.8,
         bagging_freq=1, seed=0, verbose=-1)
b = lgb.train(p, lgb.Dataset(fit[feats], fit['y']), num_boost_round=300)

print('árboles | costo val (min_costo) | % revisión | AUC falla vs sana')
for k in [2, 10, 25, 50, 100, 200, 300]:
    P = b.predict(Xv, num_iteration=k)
    yhat = utils.decide_min_cost(P)
    auc = roc_auc_score(yv > 0, 1 - P[:, 0])
    print(f'{k:7d} | {utils.total_cost(yv, yhat):21d} | {100*(yhat>0).mean():9.1f} | {auc:.3f}')

# --- 2. ¿Se distinguen los cortes de train de los puntos de validation?
adv = pd.concat([fit[feats].assign(es_val=0), Xv.assign(es_val=1)], ignore_index=True)
adv = adv.astype({c: 'category' for c in feats if c.startswith('Spec_')})
idx = np.random.default_rng(0).permutation(len(adv))
tr_i, te_i = idx[: len(idx) // 2], idx[len(idx) // 2:]
a = lgb.train(dict(objective='binary', verbose=-1, seed=0),
              lgb.Dataset(adv.loc[tr_i, feats], adv.loc[tr_i, 'es_val']), num_boost_round=200)
print('\nAUC train vs validation:', round(roc_auc_score(adv.loc[te_i, 'es_val'],
                                                         a.predict(adv.loc[te_i, feats])), 3))
imp = pd.Series(a.feature_importance('gain'), index=feats).sort_values(ascending=False)
print('Variables que más los distinguen:\n', imp.head(10).round(0).to_string())

In [ ]:
cols = ['272_0', '272_3', '272_4']

print('=== 1. CSV crudo (acumulados) ===')
for split in ['train', 'validation']:
    raw = utils.load_raw(split, 'operational_readouts')
    print(f'\n{split}:')
    print(raw[cols].describe().round(1).to_string())
    print('  % de ceros:', (raw[cols] == 0).mean().mul(100).round(1).to_dict())
    del raw

print('\n=== 2. Tasas d_ (01_datos) ===')
for split in ['train', 'validation']:
    ro = pd.read_parquet(utils.PROCESSED / f'{split}_readouts.parquet', columns=[f'd_{c}' for c in cols] + ['dt'])
    print(f'\n{split}:')
    print(ro.describe().round(3).to_string())
    del ro

print('\n=== 3. Resumen de los cortes (03_modelos) ===')
comp = pd.DataFrame({'train': fit['d_272_0_mean'].describe(), 'validation': Xv['d_272_0_mean'].describe()})
print(comp.round(3).to_string())

Ese AUC de 1 es preocupante tengo que revisar si hay fuga de información entre el train y el validation.  El modelo distingue con precisión perfecta si una fila viene de train o de validation, y lo hace casi solo con una variable: d_272_0_mean. Su importancia es 8 veces la de la siguiente.

In [ ]:
cols = ['272_0', '272_3', '272_4']

print('=== 1. CSV crudo (acumulados) ===')
for split in ['train', 'validation']:
    raw = utils.load_raw(split, 'operational_readouts')
    print(f'\n{split}:')
    print(raw[cols].describe().round(1).to_string())
    print('  % de ceros:', (raw[cols] == 0).mean().mul(100).round(1).to_dict())
    del raw

print('\n=== 2. Tasas d_ (01_datos) ===')
for split in ['train', 'validation']:
    ro = pd.read_parquet(utils.PROCESSED / f'{split}_readouts.parquet', columns=[f'd_{c}' for c in cols] + ['dt'])
    print(f'\n{split}:')
    print(ro.describe().round(3).to_string())
    del ro

print('\n=== 3. Resumen de los cortes (03_modelos) ===')
comp = pd.DataFrame({'train': fit['d_272_0_mean'].describe(), 'validation': Xv['d_272_0_mean'].describe()})
print(comp.round(3).to_string())

La diferencia viene del dataset, no de nuestro código. En el CSV crudo, la variable 272 de validation es unas 7,5 veces más pequeña que la de train: la media es 9,1 millones en train y 1,2 millones en validation, y las medianas son 7,0 millones contra 0,9 millones. Nuestro procesamiento solo arrastra esa diferencia hasta las tasas d_ y el resumen. Y dt es casi idéntico en ambos, así que el muestreo en el tiempo no es la causa.

Que sea un factor tan grande y tan parejo en todos los cuantiles sugiere algo de registro o de escala en el dataset, no un cambio real en cómo se usan los camiones. Antes de decidir qué hacer hay que responder dos preguntas:

- ¿Pasa solo con la 272 o también con otras variables?
- ¿Test se parece a validation o a train?

In [ ]:
d_cols = [c for c in pd.read_parquet(utils.PROCESSED / 'train_readouts.parquet', columns=None).columns
          if c.startswith('d_')] if False else None  # (no cargar todo dos veces)

import pyarrow.parquet as pq
d_cols = [c for c in pq.read_schema(utils.PROCESSED / 'train_readouts.parquet').names if c.startswith('d_')]

med = {}
for split in ['train', 'validation', 'test']:
    ro = pd.read_parquet(utils.PROCESSED / f'{split}_readouts.parquet', columns=d_cols)
    med[split] = ro.median()
    del ro

med = pd.DataFrame(med)
med['val/train'] = med['validation'] / med['train']
med['test/train'] = med['test'] / med['train']
med['var'] = med.index.str.extract(r'd_(\d+)_', expand=False)

print('Razón de medianas por variable (1 = igual a train):')
print(med.groupby('var')[['val/train', 'test/train']].median().round(2).to_string())

print('\nTasas d_ con razón fuera de [0.5, 2]:')
out = med[(med['val/train'] - 1).abs().gt(0.5) | (med['test/train'] < 0.5) | (med['test/train'] > 2)]
print(out[['train', 'validation', 'test', 'val/train', 'test/train']].round(2).to_string())

Opción 1 (quedarnos solo con variables que no dependen de la escala), por tres razones:

- Es fácil de defender ante un jurado. No usa nada de test, y la justificación cabe en una frase: «las proporciones no cambian si se multiplica todo el histograma».
- No hay que recalcular nada. Las columnas que necesitamos (p_, dt, los contadores estables) ya están en los archivos *_features. Solo cambiamos qué columnas recibe el modelo.
- Se puede comprobar. Con la prueba adversarial vemos si el problema desaparece o si queda alguna variable contaminada.

Lo que se pierde es la intensidad de uso de los histogramas (cuánto se usa el camión, no solo en qué condiciones)

el conjunto de variables invariantes

In [ ]:
STABLE_COUNTERS = ['171_0', '835_0', '427_0']

def is_invariant(c):
    if c == T or c.startswith('Spec_'):
        return True                                    # edad y especificaciones
    if c.startswith(('p_', 'dt_')):
        return True                                    # proporciones e intervalos
    return any(c in (f'{k}_last', f'd_{k}_last', f'd_{k}_mean', f'd_{k}_slope')
               for k in STABLE_COUNTERS)               # contadores estables

feats_inv = [c for c in feats if is_invariant(c)]
print(len(feats), '→', len(feats_inv), 'variables')

la prueba adversarial (train contra validation y train contra test)

In [ ]:
def adversarial_auc(A, B, cols, seed=0):
    X = pd.concat([A[cols], B[cols]], ignore_index=True)
    X = X.astype({c: 'category' for c in cols if c.startswith('Spec_')})
    y = np.r_[np.zeros(len(A)), np.ones(len(B))]
    idx = np.random.default_rng(seed).permutation(len(X))
    tr, te = idx[: len(idx) // 2], idx[len(idx) // 2:]
    m = lgb.train(dict(objective='binary', verbose=-1, seed=seed),
                  lgb.Dataset(X.iloc[tr], y[tr]), num_boost_round=200)
    auc = roc_auc_score(y[te], m.predict(X.iloc[te]))
    imp = pd.Series(m.feature_importance('gain'), index=cols).sort_values(ascending=False)
    return auc, imp

Ft = utils.load_processed('test_features')
for name, other in [('validation', Xv), ('test', Ft)]:
    for label, cols in [('todas', feats), ('invariantes', feats_inv)]:
        auc, imp = adversarial_auc(fit, other, cols)
        print(f'train vs {name:10s} | {label:11s} | AUC = {auc:.3f}')
    print('   las que más delatan (invariantes):', list(imp.head(5).index))

reentrenar con las variables invariantes

In [ ]:
# 1. AUC de cada variable sola (0,5 = no delata nada; cerca de 0 o 1 = delata mucho)
num = [c for c in feats_inv if not c.startswith('Spec_')]
y_adv = np.r_[np.zeros(len(fit)), np.ones(len(Xv))]
uni = {}
for c in num:
    x = np.r_[fit[c].to_numpy(), Xv[c].to_numpy()]
    m = ~np.isnan(x)
    if m.sum() > 100 and len(np.unique(y_adv[m])) == 2:
        a = roc_auc_score(y_adv[m], x[m])
        uni[c] = max(a, 1 - a)
uni = pd.Series(uni).sort_values(ascending=False)

print('Variables más desplazadas (AUC individual):')
print(uni.head(15).round(3).to_string())
print('\nCuántas variables superan cada nivel:')
for t in [0.55, 0.6, 0.7, 0.8, 0.9]:
    print(f'  > {t}: {(uni > t).sum()} de {len(uni)}')

# 2. ¿Los cortes están en el mismo punto de la historia?
def n_hist(split, pts):
    ro = pd.read_parquet(utils.PROCESSED / f'{split}_readouts.parquet', columns=[ID, T])
    ro['n_hist'] = ro.groupby(ID).cumcount() + 1
    ro['n_total'] = ro.groupby(ID)[T].transform('size')
    return pts[[ID, T]].merge(ro, on=[ID, T])

h_tr = n_hist('train', fit)
h_va = n_hist('validation', Fv)
print('\nLecturas antes del corte (n_hist):')
print(pd.DataFrame({'train': h_tr['n_hist'].describe(), 'validation': h_va['n_hist'].describe()}).round(1).to_string())
print('\nEdad en el corte (time_step):')
print(pd.DataFrame({'train': fit[T].describe(), 'validation': Xv[T].describe()}).round(1).to_string())

La causa principal no es el escalamiento: nuestros cortes de train están en una etapa más temprana de la vida del camión que los de validation.elegimos el corte al azar entre todas las lecturas de cada camión. El challenge, en cambio, eligió cortes más tardíos. Encaja con algo que ya vimos: en validation hay más camiones a punto de fallar (clase 4), que por definición están al final de su historia.

Procedemos a probar dar peso a los cortes.

In [ ]:
h_te = n_hist('test', Ft)
print('Mediana lecturas antes del corte  | train', h_tr['n_hist'].median(),
      '| val', h_va['n_hist'].median(), '| test', h_te['n_hist'].median())
print('Mediana edad en el corte          | train', fit[T].median(),
      '| val', Xv[T].median(), '| test', Ft[T].median())

# Modelo que estima qué tan "tipo validation" es cada corte, usando solo edad y n.º de lecturas
pos_cols = [T, 'n_hist']
A = pd.concat([data['landmarking'][[ID, T]].merge(lm[[ID, T, 'n_hist']], on=[ID, T]),
               data['convencional'][[ID, T]].merge(conv[[ID, T, 'n_hist']], on=[ID, T])]).drop_duplicates()
B = h_va[[T, 'n_hist']]
Xw = pd.concat([A[pos_cols], B[pos_cols]], ignore_index=True)
yw = np.r_[np.zeros(len(A)), np.ones(len(B))]
mw = lgb.train(dict(objective='binary', num_leaves=8, min_data_in_leaf=200, verbose=-1, seed=0),
               lgb.Dataset(Xw, yw), num_boost_round=100)

def position_weights(df):
    """Peso = densidad(validation) / densidad(train) en (edad, n.º de lecturas)."""
    p = np.clip(mw.predict(df[pos_cols]), 1e-3, 1 - 1e-3)
    w = p / (1 - p)
    w = np.minimum(w, np.quantile(w, 0.99))      # recorta pesos extremos
    return w / w.mean()

for name, lab in [('convencional', conv), ('landmarking', lm)]:
    data[name] = data[name].drop(columns=['n_hist', 'w'], errors='ignore') \
                           .merge(lab[[ID, T, 'n_hist']], on=[ID, T])
    data[name]['w'] = position_weights(data[name])
    print(name, '| peso: min', data[name]['w'].min().round(3),
          '| mediana', data[name]['w'].median().round(3), '| máx', data[name]['w'].max().round(2))

# Comprobación: edad media de train con y sin pesos frente a validation
d = data['convencional']
print('\nEdad media | train sin pesos', d[T].mean().round(1),
      '| con pesos', np.average(d[T], weights=d['w']).round(1), '| validation', Xv[T].mean().round(1))

Todo cuadra con lo que esperábamos:

- Test se parece a validation: mediana de 34 lecturas en ambos, y edad de 171,6 contra 169,4. El challenge eligió los cortes de la misma forma en los dos. Por eso, la corrección que ajustemos con validation también sirve para la evaluación final en test.
- Los pesos están en un rango sano: van de 0,03 a 5,3. Ningún corte de train pesa más de 5 veces el promedio, así que ninguno domina el entrenamiento.
- La edad media se corrige, aunque se pasa un poco: de 137 sin pesos a 206 con pesos, cuando validation tiene 191. La media es sensible a los camiones más viejos. Antes de afinar los pesos, veamos si el modelo ya transfiere.

In [ ]:
#  VERIFICAMOS

#SAQUEMOS EL ACCURACY

from sklearn.metrics import accuracy_score
res_w = []
for name, d in data.items():
    fit_d = d[~d[ID].isin(cal_ids)]
    for pesos, w in [('sin pesos', None), ('con pesos', fit_d['w'].to_numpy())]:
        b = utils.train_lgbm(fit_d[feats_inv], fit_d['y'], Xv[feats_inv], yv, w_fit=w)
        P = b.predict(Xv[feats_inv], num_iteration=b.best_iteration)
        for rule, decide in [('argmax', utils.decide_argmax), ('min_costo', utils.decide_min_cost)]:
            yhat = decide(P)
            res_w.append({'formulación': name, 'pesos': pesos, 'regla': rule,
                          'costo_total': utils.total_cost(yv, yhat),
                          '% a revisión': round(100 * (yhat > 0).mean(), 1),
                          'iteraciones': b.best_iteration,
                          'AUC falla': round(roc_auc_score(yv > 0, 1 - P[:, 0]), 3)
                          })
res_w = pd.DataFrame(res_w)
res_w.to_csv(utils.RESULTS / 'lgbm_invariantes_pesos.csv', index=False)
res_w

Ahora sí funciona. Las dos correcciones juntas, variables invariantes y pesos por posición, hacen que el modelo aprenda algo que transfiere.

- Landmarking con pesos es lo mejor hasta ahora: se detiene en 271 árboles (antes eran 2), con AUC de 0,747 y costo de 37 654. Para eso manda a revisión al 35 % de la flota. Es un 24 % menos que «siempre 4» (49 566) y está en el nivel de Ferrisi en validation (36 595).
- En la convencional, los pesos ayudan poco: sin pesos cuesta 37 884 mandando al 40 % a revisión; con pesos cuesta 40 462 mandando al 22 %. Con un solo corte por camión, el desfase de edad pesa menos.
- Argmax nunca sirve: siempre predice clase 0, porque las fallas son raras. Ya es una primera evidencia a favor de H2: con la misma probabilidad, la regla de mínimo costo esperado ahorra entre 17 000 y 20 000.

In [ ]:
lm = utils.load_processed('train_labels_landmark')       # ya trae n_hist
va_pts = utils.add_position('validation', utils.load_processed('validation_eval'))

weights = lm[[ID, T, 'n_hist']].copy()
weights['w'] = utils.position_weights(weights, va_pts)
utils.to_parquet(weights, 'train_weights')

print(weights.shape)
print(weights['w'].describe().round(3).to_string())
print('\nEdad media | sin pesos', weights[T].mean().round(1),
      '| con pesos', np.average(weights[T], weights=weights['w']).round(1),
      '| validation', va_pts[T].mean().round(1))

Los pesos quedaron muy bien: la edad media de train con pesos es 191,4, prácticamente la misma de validation (191,1). La versión anterior se pasaba hasta 206, así que esta corrige mejor.

Además, los pesos son moderados: van de 0,05 a 3,3, y ningún corte pesa más de 3,3 veces el promedio. Por la dispersión de los pesos, el entrenamiento rinde como si hubiera cerca del 63 % de los cortes (unos 143 000 de 226 215)

In [ ]:
lm   = utils.load_processed('train_labels_landmark')
conv = utils.load_processed('train_labels_conv')
va_pts = utils.add_position('validation', utils.load_processed('validation_eval'))

sets = {
    'w_todos':        lm,                          # todos los landmarks (para riesgo discreto)
    'w_conv':         conv,                        # formulación convencional
    'w_lm_conocidos': lm[lm['y_cens'].notna()],    # landmarking: solo clase conocida
}

W = lm[[ID, T, 'n_hist']].copy()
for col, s in sets.items():
    s = s[[ID, T, 'n_hist']].copy()
    s[col] = utils.position_weights(s, va_pts)
    W = W.merge(s[[ID, T, col]], on=[ID, T], how='left')
    print(f'{col:15s} | filas {s.shape[0]:6d} | edad media con pesos '
          f'{np.average(s[T], weights=s[col]):.1f} (validation {va_pts[T].mean():.1f}) '
          f'| peso máx {s[col].max():.2f}')

utils.to_parquet(W, 'train_weights_por_conjunto')

In [ ]:
weights = utils.load_processed('train_weights_por_conjunto')
wcol = {'convencional': 'w_conv', 'landmarking': 'w_lm_conocidos'}

Ft = utils.load_processed('test_features').merge(
        utils.load_processed('test_eval')[[ID, T, 'class_label']], on=[ID, T])
(utils.RESULTS / 'models').mkdir(parents=True, exist_ok=True)

for form, d in data.items():
    wf = weights[[ID, T, wcol[form]]].rename(columns={wcol[form]: 'w'})
    d = d.drop(columns=['w', 'n_hist'], errors='ignore').merge(wf, on=[ID, T])
    fit_d = d[~d[ID].isin(cal_ids)].copy()
    cal_d = d[d[ID].isin(cal_ids)].copy()
    for part in (fit_d, cal_d):
        part['w'] = part['w'] / part['w'].mean()

    b = utils.train_lgbm(fit_d[feats_inv], fit_d['y'], Xv[feats_inv], yv, w_fit=fit_d['w'])
    b.save_model(str(utils.RESULTS / 'models' / f'lgbm_{form}.txt'), num_iteration=b.best_iteration)

    for split, df, y, w in [('cal', cal_d, cal_d['y'], cal_d['w']),
                            ('validation', Fv, Fv['class_label'], None),
                            ('test', Ft, Ft['class_label'], None)]:
        P = b.predict(df[feats_inv], num_iteration=b.best_iteration)
        utils.save_predictions('lgbm', form, split, df, P, y, w)
    print(form, '| árboles:', b.best_iteration, '| calibración:', len(cal_d), 'cortes')

In [ ]:
from sklearn.metrics import roc_auc_score

F = utils.load_processed('train_features')[[ID] + feats_inv]
W = utils.load_processed('train_weights_por_conjunto')

# formulación: (etiquetas, columna de pesos, ¿censurados = sobreviven?, etiqueta para calibrar)
haz_sets = {
    'convencional': (conv, 'w_conv',  True,  'y_conv'),
    'landmarking':  (lm,   'w_todos', False, 'y_cens'),   # usa también los cortes censurados
}

for form, (lab, wc, cas, ycol) in haz_sets.items():
    d = (lab[[ID, T, 'time_left', 'event', ycol]]
         .merge(F, on=[ID, T])
         .merge(W[[ID, T, wc]].rename(columns={wc: 'w'}), on=[ID, T]))
    is_cal = d[ID].isin(cal_ids)
    fit_d = d[~is_cal]
    cal_d = d[is_cal & d[ycol].notna()]

    pre = utils.TabularPreprocessor().fit(fit_d, feats_inv)
    R, E = utils.hazard_targets(fit_d['time_left'], fit_d['event'], censor_as_survival=cas)
    m = utils.DiscreteHazard(l2=1.0).fit(pre.transform(fit_d), R, E,
                                         w=fit_d['w'] / fit_d['w'].mean())

    for split, df, y, w in [('cal', cal_d, cal_d[ycol].astype(int), cal_d['w'] / cal_d['w'].mean()),
                            ('validation', Fv, Fv['class_label'], None),
                            ('test', Ft, Ft['class_label'], None)]:
        utils.save_predictions('hazard', form, split, df, m.predict_proba(pre.transform(df)), y, w)

    Pv = m.predict_proba(pre.transform(Fv))
    print(f'{form:12s} | filas {len(fit_d):6d} | iteraciones {m.res_.nit:3d} | '
          f'convergió: {m.res_.success} | AUC falla val {roc_auc_score(yv > 0, 1 - Pv[:, 0]):.3f}')

In [ ]:
search = []
for form, (lab, wc, cas, ycol) in haz_sets.items():
    d = (lab[[ID, T, 'time_left', 'event', ycol]]
         .merge(F, on=[ID, T])
         .merge(W[[ID, T, wc]].rename(columns={wc: 'w'}), on=[ID, T]))
    fit_d = d[~d[ID].isin(cal_ids)]
    pre = utils.TabularPreprocessor().fit(fit_d, feats_inv)
    Xf, Xval = pre.transform(fit_d), pre.transform(Fv)
    R, E = utils.hazard_targets(fit_d['time_left'], fit_d['event'], censor_as_survival=cas)
    w = fit_d['w'] / fit_d['w'].mean()

    for l2 in [1, 10, 100, 1_000, 10_000, 100_000]:
        m = utils.DiscreteHazard(l2=l2).fit(Xf, R, E, w=w)
        Pv = m.predict_proba(Xval)
        theta, c_umbral = utils.best_threshold(Pv, yv)
        search.append({'formulación': form, 'l2': l2, 'iteraciones': m.res_.nit,
                       'AUC falla': round(roc_auc_score(yv > 0, 1 - Pv[:, 0]), 3),
                       'costo min_costo': utils.total_cost(yv, utils.decide_min_cost(Pv)),
                       'costo umbral': c_umbral})
        print(search[-1])

search = pd.DataFrame(search)
search.to_csv(utils.RESULTS / 'hazard_busqueda_l2.csv', index=False)
search

La hipótesis

En validation, cada camión tiene una sola etiqueta para su último corte. El artículo del dataset define las clases como el tiempo antes de la falla, pero no explica qué etiqueta reciben los camiones censurados, los que no fallaron mientras duró el estudio. Lo más probable es que reciban clase 0, porque no hay otra opción. Si es así, la etiqueta «correcta» del examen ya trae la suposición de la formulación convencional: un camión censurado cerca del fin del estudio cuenta como sano, aunque haya fallado una semana después.

- La formulación convencional aprende exactamente esa convención, y por eso le va bien en el examen.
- La formulación con censura aprende la probabilidad real de falla, que para esos camiones es mayor. Pero el examen los califica como sanos, así que el modelo «correcto» queda castigado por alertar sobre camiones que el examen da por buenos.

Si se confirma, es un hallazgo importante para H1: con este benchmark no se puede premiar un tratamiento correcto de la censura, porque las etiquetas de evaluación la ignoran. Eso no prueba que la formulación con censura sea peor para una flota real. Prueba que el benchmark mide otra cosa.

In [ ]:
best_l2 = {'convencional': 10, 'landmarking': 100}

for form, (lab, wc, cas, ycol) in haz_sets.items():
    d = (lab[[ID, T, 'time_left', 'event', ycol]]
         .merge(F, on=[ID, T])
         .merge(W[[ID, T, wc]].rename(columns={wc: 'w'}), on=[ID, T]))
    is_cal = d[ID].isin(cal_ids)
    fit_d, cal_d = d[~is_cal], d[is_cal & d[ycol].notna()]

    pre = utils.TabularPreprocessor().fit(fit_d, feats_inv)
    R, E = utils.hazard_targets(fit_d['time_left'], fit_d['event'], censor_as_survival=cas)
    m = utils.DiscreteHazard(l2=best_l2[form]).fit(pre.transform(fit_d), R, E,
                                                   w=fit_d['w'] / fit_d['w'].mean())

    for split, df, y, w in [('cal', cal_d, cal_d[ycol].astype(int), cal_d['w'] / cal_d['w'].mean()),
                            ('validation', Fv, Fv['class_label'], None),
                            ('test', Ft, Ft['class_label'], None)]:
        utils.save_predictions('hazard', form, split, df, m.predict_proba(pre.transform(df)), y, w)

    Pv = m.predict_proba(pre.transform(Fv))
    print(f'{form:12s} | l2={best_l2[form]:>4} | P(falla) media en val: {(1 - Pv[:, 0]).mean():.4f} '
          f'| por clase: {Pv[:, 1:].mean(0).round(4)}')

print(f'\nTasa real de falla en validation: {(yv > 0).mean():.4f} '
      f'| por clase: {np.bincount(yv, minlength=5)[1:] / len(yv)}')

- Si landmarking predice claramente más falla que la convencional y que el 2,7 % real, la hipótesis se sostiene. El modelo con censura «ve» fallas que el examen no cuenta.
- Si ambos predicen parecido, la hipótesis cae y habría que buscar otra explicación.

In [ ]:
W = utils.load_processed('train_weights_por_conjunto')
wcol = {'convencional': 'w_conv', 'landmarking': 'w_lm_conocidos'}

ord_search = []
for form, d in data.items():
    wf = W[[ID, T, wcol[form]]].rename(columns={wcol[form]: 'w'})
    d = d.drop(columns=['w', 'n_hist'], errors='ignore').merge(wf, on=[ID, T])
    fit_d = d[~d[ID].isin(cal_ids)]
    cal_d = d[d[ID].isin(cal_ids)]

    pre = utils.TabularPreprocessor().fit(fit_d, feats_inv)
    Xf, Xval = pre.transform(fit_d), pre.transform(Fv)
    w = fit_d['w'] / fit_d['w'].mean()

    fitted = {}
    for l2 in [1, 10, 100, 1_000, 10_000]:
        m = utils.OrdinalLogistic(l2=l2).fit(Xf, fit_d['y'], w)
        Pv = m.predict_proba(Xval)
        fitted[l2] = m
        ord_search.append({'formulación': form, 'l2': l2, 'iteraciones': m.res_.nit,
                           'convergió': m.res_.success,
                           'AUC falla': round(roc_auc_score(yv > 0, 1 - Pv[:, 0]), 3),
                           'costo min_costo': utils.total_cost(yv, utils.decide_min_cost(Pv)),
                           'costo umbral': utils.best_threshold(Pv, yv)[1]})
        print(ord_search[-1])

    # mismo criterio que en riesgo discreto: menor costo con mínimo costo esperado
    res_f = [r for r in ord_search if r['formulación'] == form]
    best = min(res_f, key=lambda r: r['costo min_costo'])['l2']
    m = fitted[best]
    print(f'→ {form}: l2 elegido = {best}\n')

    for split, df, y, ww in [('cal', cal_d, cal_d['y'], cal_d['w'] / cal_d['w'].mean()),
                             ('validation', Fv, Fv['class_label'], None),
                             ('test', Ft, Ft['class_label'], None)]:
        utils.save_predictions('ordinal', form, split, df, m.predict_proba(pre.transform(df)), y, ww)

ord_search = pd.DataFrame(ord_search)
ord_search.to_csv(utils.RESULTS / 'ordinal_busqueda_l2.csv', index=False)
ord_search